# W12 Practice — An MCP Server of Our Own

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week12/W12_practice_mcp_server.ipynb)

**Goal.** Build an MCP server over a web API and give its tools to an agent. The server sends the requests to the API and returns data; the model of the agent chooses the calls and writes the answer. Two questions run on two servers over the same API: the first server has one tool for each request of the API, the second one tool for each task of the user. Part 5 takes a server written by others and finds its design from what a client sees.

| Part | What you do | Main names |
|---|---|---|
| 1. The recipe API | see the eleven requests of the API and send two of them by hand | `httpx2` |
| 2. A server that wraps the API | write a server with one tool for each request, give its tools to an agent, and run two questions | `MCPServer`, `@mcp.tool`, `MCPAdapter`, `create_agent` |
| 3. Tools from the tasks of the user | define the tasks of the user and one tool for each, by two principles of Anthropic | |
| 4. A server built from the tasks | write the second server and run the same questions | `ToolError` |
| 5. Reverse engineering a server | read the definitions, the results, and the trace of the KOSIS server, and write down its design | `ainvoke` |

**Cell labels.** The label is the first line of a code cell, or the second line when the cell saves a file with `%%writefile`:

- `# ▶ RUN · example`: read each line, then run.
- `# ▶ RUN · setup`: just run.

## An MCP Server over a Web API

An MCP server gives a web API the form of tools. Three programs take part. The server only answers the calls that it gets; the host decides which tools to call and writes the answer.

| Program | What it does | Where |
|---|---|---|
| Web API | keeps the recipes and answers HTTP requests | TheMealDB, on the internet |
| MCP server | offers tools; a tool sends requests to the web API and returns the data | a file in this runtime, started as its own program |
| MCP host | gives the tools to a model; the model chooses the calls and writes the answer | this notebook |

A server in Python is written in five steps.

| Step | Code |
|---|---|
| 1. Make the server object | `mcp = MCPServer("recipes")` |
| 2. Write one function for each tool | `@mcp.tool()` above the function |
| 3. Call the web API in the function and return the result | `httpx2.get(...)` |
| 4. Run the server | `mcp.run()` |
| 5. Connect a client and get the tools | `MCPAdapter(PythonStdioTransport("server.py"))`, `list_tools()` |

The server makes the definition of a tool from the function: the function name becomes the tool name, the type hints the arguments, and the docstring the description. The client starts the server file as a program and talks to it through its standard input and output (the **stdio** transport of MCP).

Source of the steps: MCP documentation, *Build an MCP server*.

## 0. Setup

*Do:* run the two cells; paste your API key into the second.

In [ ]:
# ▶ RUN · setup: just run
%pip install -q "langchain[mcp]==1.4.4" "langchain-openai==1.7.0" "langgraph==1.2.14"

In [ ]:
# ▶ RUN · setup: just run
import os
from langchain.chat_models import init_chat_model

OPENAI_API_KEY = "PASTE-YOUR-KEY-HERE"   # paste your key between the quotes
if OPENAI_API_KEY != "PASTE-YOUR-KEY-HERE":
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY   # LangChain reads the key from here

MODEL = "openai:gpt-5.4"
model = init_chat_model(MODEL)                     # one model object for the whole notebook

## 1. The Recipe API

TheMealDB is a free database of recipes with a web API that needs no sign-up. With the public test key `1`, the API has eleven requests (source: themealdb.com/api.php). Each request takes at most one condition.

| Request | What it returns |
|---|---|
| `search.php?s=<name>` | the recipes whose name contains the text |
| `search.php?f=<letter>` | the recipes whose name begins with the letter |
| `lookup.php?i=<meal ID>` | the recipe of one meal |
| `random.php` | the recipe of one meal, chosen at random |
| `filter.php?i=<ingredient>` | the meals that use one ingredient (name, picture link, ID) |
| `filter.php?c=<category>` | the meals of one category, for example Seafood |
| `filter.php?a=<area>` | the meals of one area, for example Italian |
| `categories.php` | the 14 categories with a picture and a description |
| `list.php?c=list` | the names of the 14 categories |
| `list.php?a=list` | the names of the 195 areas |
| `list.php?i=list` | the 992 ingredients with a description |

*Do:* run the cell. It sends two of the requests by hand and prints the replies.

In [ ]:
# ▶ RUN · example: read each line, then run
import httpx2                                      # an HTTP client for Python; it came with the packages of the setup

API = "https://www.themealdb.com/api/json/v1/1"    # the address of the API; "1" is the public test key

reply = httpx2.get(API + "/filter.php", params={"i": "chicken breast"})   # the meals that use chicken breast
meals = reply.json()["meals"]                      # .json() turns the JSON of the reply into Python lists and dicts
print(len(meals), "meals; the first one:", meals[0])

reply = httpx2.get(API + "/lookup.php", params={"i": meals[0]["idMeal"]})   # the recipe of the first meal, by its ID
print(reply.text)

**Reading the output.** A filter returns a short entry for each meal: the name, a picture link, and the ID (the filter by ingredient also gives the area). A recipe needs a second request; its reply has 54 fields, among them twenty pairs `strIngredient1` to `strIngredient20` and `strMeasure1` to `strMeasure20`, and the pairs that a meal does not use are empty.

## 2. A Server that Wraps the API

The first server has one tool for each of the eleven requests and returns the reply of the API as it comes.

*Do:* run the cell. It saves the server as `meal_mcp_v1.py`.

In [ ]:
%%writefile meal_mcp_v1.py
# ▶ RUN · example: read each line, then run
import httpx2
from mcp.server import MCPServer

API = "https://www.themealdb.com/api/json/v1/1"

## Step 1 - the server object
mcp = MCPServer("recipes", log_level="WARNING")    # the name that the server gives to its clients; log only warnings


## Steps 2 and 3 - one function for each request of the API; the reply of the API is the tool result
@mcp.tool()
def search_by_name(name: str) -> str:
    """Find the recipes whose name contains the given text, for example "lasagne"."""
    return httpx2.get(API + "/search.php", params={"s": name}).text


@mcp.tool()
def search_by_first_letter(letter: str) -> str:
    """Find the recipes whose name begins with the given letter."""
    return httpx2.get(API + "/search.php", params={"f": letter}).text


@mcp.tool()
def get_recipe(meal_id: str) -> str:
    """Get the recipe of one meal by its ID, for example "52772"."""
    return httpx2.get(API + "/lookup.php", params={"i": meal_id}).text


@mcp.tool()
def random_meal() -> str:
    """Get the recipe of one meal chosen at random."""
    return httpx2.get(API + "/random.php").text


@mcp.tool()
def filter_by_ingredient(ingredient: str) -> str:
    """Find the meals that use one ingredient, for example "chicken breast"."""
    return httpx2.get(API + "/filter.php", params={"i": ingredient}).text


@mcp.tool()
def filter_by_category(category: str) -> str:
    """Find the meals of one category, for example "Seafood"."""
    return httpx2.get(API + "/filter.php", params={"c": category}).text


@mcp.tool()
def filter_by_area(area: str) -> str:
    """Find the meals of one area, for example "Italian"."""
    return httpx2.get(API + "/filter.php", params={"a": area}).text


@mcp.tool()
def get_categories() -> str:
    """Get all categories with a picture and a description."""
    return httpx2.get(API + "/categories.php").text


@mcp.tool()
def list_category_names() -> str:
    """Get the names of all categories."""
    return httpx2.get(API + "/list.php", params={"c": "list"}).text


@mcp.tool()
def list_area_names() -> str:
    """Get the names of all areas."""
    return httpx2.get(API + "/list.php", params={"a": "list"}).text


@mcp.tool()
def list_ingredients() -> str:
    """Get all ingredients with a description."""
    return httpx2.get(API + "/list.php", params={"i": "list"}).text


## Step 4 - run the server
if __name__ == "__main__":                         # true when this file is started as a program
    mcp.run()                                      # stdio: read requests from standard input, write replies to standard output

### 2.2 Two questions on this server

The two questions in `QUESTIONS` are what a user of a recipe service would ask: one gives ingredients, the other a cuisine and a kind of meal.

*Do:* run the cell. It connects to the server (step 5), prints the number of tools and the length of their definitions (the model gets them with every call), gives the tools to an agent, and prints the path of each run: each tool call, the length of each tool result, the totals, and the answer.

In [ ]:
# ▶ RUN · example: read each line, then run
from pathlib import Path
from fastmcp.client.transports import PythonStdioTransport
from langchain.mcp import MCPAdapter               # this import prints one warning: this part of LangChain is in beta
from langchain.agents import create_agent
from langchain.messages import HumanMessage

def show(result):
    """Print the path of one run: the tool calls, the length of each tool result, the totals, and the answer."""
    calls = 0
    characters = 0
    for message in result["messages"]:
        if message.type == "ai":
            for call in message.tool_calls:        # one AI message can ask for several tools
                print("TOOL CALL  :", call["name"], call["args"])
                calls = calls + 1
        if message.type == "tool":
            print("TOOL RESULT:", message.name, "-", len(message.text), "characters")
            characters = characters + len(message.text)
    print("IN ALL     :", calls, "calls,", characters, "characters of tool results")
    print()
    print(result["messages"][-1].text)             # the last message is the answer
    print()

def show_tools(tools):
    """Print the number of tools and the length of their definitions: name, description, and arguments."""
    length = 0
    for tool in tools:
        length = length + len(tool.name) + len(tool.description) + len(str(tool.args))
    print(len(tools), "tools,", length, "characters of definitions")
    print()

## Step 5 - connect a client and get the tools
server_v1 = PythonStdioTransport("meal_mcp_v1.py", log_file=Path("server.log"))   # start the file as a program, talk over stdio;
                                                   # its messages go to server.log (Colab has no place to show them)
async with MCPAdapter(server_v1) as adapter:
    tools_v1 = await adapter.list_tools()
show_tools(tools_v1)

agent_v1 = create_agent(model, tools_v1)

QUESTIONS = [
    "냉장고에 닭가슴살이랑 마늘이 있어. 저녁으로 뭘 해 먹을 수 있을지 두 가지 추천하고 만드는 법도 알려줘.",
    "이탈리아 해산물 요리는 뭐가 있어?",
]
for question in QUESTIONS:
    print("=====", question)
    result = await agent_v1.ainvoke({"messages": [HumanMessage(question)]})   # the tools are asynchronous, so the agent runs with ainvoke
    show(result)

**Reading the output.** In our run the model combined the lists of the API itself. For the first question it called `filter_by_ingredient` twice and read 3,323 and 42,111 characters to find the meals that are in both lists, then opened two recipes: 4 calls and 51,136 characters. The first dish of its answer comes from one of the two recipes; the second dish (마늘 닭가슴살 볶음) is not a meal of the API, and the second recipe that it opened (General Tsos Chicken) is not in the answer. For the second question it read the Italian meals (3,830 characters) and the Seafood meals (15,725 characters): 2 calls and 19,555 characters. It listed three meals, and one of them, Chilli prawn linguine, is in the category Pasta, not Seafood.

## 3. Tools from the Tasks of the User

The server of Part 2 was built from the list of requests of the API. A second server is built from the other side: first the tasks that a user of a recipe service wants done, then one tool for each task. Anthropic's guide for writers of tools gives two principles for this.

### One call for one task of the user

The questions that a user asks of this API are of two tasks:

| Task of the user | What the user gives | What the user wants back |
|---|---|---|
| find meals | conditions: ingredients, a category, an area, a part of the name | the meals that meet all the conditions |
| make one meal | the meal | its ingredients with their measures and its instructions |

A tool for a task sends all the requests that the task needs and combines their results, so that the model does not chain the requests and read the results in between. The guide: "Tools can consolidate functionality, handling potentially multiple discrete operations (or API calls) under the hood." The requests of the API go into the two tools as follows:

- `find_meals` sends one filter or name search for each condition and keeps the meals that are in every list.
- `get_recipe` sends the lookup.
- The lists of names become no tool of their own: the 14 categories go into the type of the argument, and the server checks the area against the 195 areas. The guide warns against a tool that returns a whole list for the model to read: "if an LLM agent uses a tool that returns ALL contacts and then has to read through each one token-by-token, it's wasting its limited context space on irrelevant information".
- The search by first letter and the random meal serve neither task, so they become no tool.

### A small result

A tool returns the fields that its task needs and a limited number of items, not the raw reply of the API. The guide: "return only high signal information back to agents", with "pagination, range selection, filtering, and/or truncation with sensible default parameter values".

Both principles move work on data from the model to the server: the server searches, combines, and trims. The model still chooses the conditions and the meals and writes the answer.

Source: Anthropic, *Writing effective tools for agents* (2025), sections "Choosing the right tools for agents", "Returning meaningful context from your tools", and "Optimizing tool responses for token efficiency".

## 4. A Server Built from the Tasks

The second server has the two tools of Part 3. Both return a dictionary with only the fields that the task needs (a small result); the server sends it to the client as JSON text. When an area is not one of the 195 areas, `find_meals` raises a `ToolError` with the closest names, so that the model can correct its call.

*Do:* run the cell. It saves the server as `meal_mcp_v2.py`.

In [ ]:
%%writefile meal_mcp_v2.py
# ▶ RUN · example: read each line, then run
from difflib import get_close_matches             # finds the most similar strings in a list
from typing import Literal
import httpx2
from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError   # an error that the client sees with its text

API = "https://www.themealdb.com/api/json/v1/1"
mcp = MCPServer("recipes", log_level="WARNING")

# the 14 categories of list.php?c=list; as a type, they are part of the definition that the model reads
Category = Literal["Beef", "Breakfast", "Chicken", "Dessert", "Goat", "Lamb", "Miscellaneous",
                   "Pasta", "Pork", "Seafood", "Side", "Starter", "Vegan", "Vegetarian"]


def get_meals(path, params):
    """Send one request to the API and return its list of meals ([] when there is none). Not a tool."""
    meals = httpx2.get(API + path, params=params).json()["meals"]
    if meals is None:
        return []
    return meals


## Task 1 - find meals: one request for each condition, then the meals that are in every list
@mcp.tool()
def find_meals(ingredients: list[str] = [], category: Category | None = None, area: str | None = None,
               name: str | None = None, limit: int = 5) -> dict:
    """Find the meals that meet all of the given conditions. Give at least one condition.

    Args:
        ingredients: Ingredients in English that every meal must use, for example ["chicken breast", "garlic"].
        category: The category of the meals.
        area: The cuisine in English, for example "Italian" or "Japanese".
        name: A part of the name of the meal, for example "lasagne".
        limit: The largest number of meals to return.
    """
    lists = []                                     # one list of meals for each condition
    for ingredient in ingredients:
        lists.append(get_meals("/filter.php", {"i": ingredient}))
    if category:
        lists.append(get_meals("/filter.php", {"c": category}))
    if area:
        areas = [item["strArea"] for item in get_meals("/list.php", {"a": "list"})]
        if area not in areas:
            raise ToolError("Unknown area " + area + ". The closest areas: " + ", ".join(get_close_matches(area, areas, n=5, cutoff=0.5)))
        lists.append(get_meals("/filter.php", {"a": area}))
    if name:
        lists.append(get_meals("/search.php", {"s": name}))
    if len(lists) == 0:
        raise ToolError("Give at least one condition.")

    found = lists[0]
    for meals in lists[1:]:
        ids = [meal["idMeal"] for meal in meals]
        found = [meal for meal in found if meal["idMeal"] in ids]   # keep the meals that are also in this list
    meals = []
    for meal in found[:limit]:                     # a small result: at most `limit` meals, two fields each
        meals.append({"name": meal["strMeal"], "meal_id": meal["idMeal"]})   # get_recipe takes the meal_id
    return {"count": len(found), "meals": meals}   # count: the number of all meals found


## Task 2 - make one meal: the recipe without the empty fields
@mcp.tool()
def get_recipe(meal_id: str) -> dict:
    """Get the recipe of one meal.

    Args:
        meal_id: The ID of the meal, for example "52772".
    """
    meal = get_meals("/lookup.php", {"i": meal_id})[0]
    ingredients = []
    for i in range(1, 21):                         # the API keeps twenty numbered pairs
        ingredient = meal["strIngredient" + str(i)]
        if ingredient:                             # skip the empty pairs
            ingredients.append({"name": ingredient, "measure": meal["strMeasure" + str(i)]})
    return {"name": meal["strMeal"], "category": meal["strCategory"], "area": meal["strArea"],   # 6 of the 54 fields
            "ingredients": ingredients, "instructions": meal["strInstructions"], "video": meal["strYoutube"]}


if __name__ == "__main__":
    mcp.run()

### 4.2 The same questions on this server

*Do:* run the cell. It connects to the second server, prints its tools and the length of their definitions, and runs `QUESTIONS` again.

In [ ]:
# ▶ RUN · example: read each line, then run
server_v2 = PythonStdioTransport("meal_mcp_v2.py", log_file=Path("server.log"))
async with MCPAdapter(server_v2) as adapter:
    tools_v2 = await adapter.list_tools()

for tool in tools_v2:
    print(tool.name, list(tool.args))
show_tools(tools_v2)

agent_v2 = create_agent(model, tools_v2)
for question in QUESTIONS:
    print("=====", question)
    result = await agent_v2.ainvoke({"messages": [HumanMessage(question)]})
    show(result)

**Reading the output.** In our run the first question took 3 calls: one `find_meals` with both ingredients and `limit` set to 2, then the two recipes, with 4,112 characters against 51,136 in Part 2. The answer gives the ingredients and the steps of the two recipes. The second question took one call with `area` and `category` and 190 characters, against 2 calls and 19,555 characters, and the answer has the two meals that are in both lists. The definitions did not become shorter: the 2 tools have 1,232 characters, the 11 tools of Part 2 had 1,040, because the 14 categories and the descriptions of the arguments are in them. What the server took over is the combination of the lists: the model no longer reads them, and it can no longer combine them wrongly.

## 5. Reverse Engineering a Server

In Parts 2 to 4 we designed a server of our own. Most servers that an agent uses are written by others. The MCP server of KOSIS (국가통계포털, the statistics portal of Statistics Korea) is one: an official pilot service on the internet, open to anyone without a key, with no published code. A client still sees three things of it: the definitions of its tools, the results of its tools, and the trace of an agent that uses it. From these three we find how its designers answered the questions that we answered for our own server:

1. Which tools does the server offer, and how is the work split among them?
2. What does the server tell the model about the use of the tools?
3. What does a result carry, and what does the server do with a call that is not exact?

Keep notes as you go; 5.4 collects them in one table.

### 5.1 The definitions

*Do:* run the cell. Then answer questions 1 and 2: write the tools in the order in which they would answer "2025년 서울과 부산의 주민등록인구", and mark the sentences that tell the model an order or a rule.

In [ ]:
# ▶ RUN · example: read each line, then run
KOSIS_URL = "https://kosismcp2026.vercel.app/api/mcp"   # the address of the server, from the KOSIS guide

async with MCPAdapter(KOSIS_URL) as adapter:       # an address instead of a file: this server runs on the internet
    kosis_tools = await adapter.list_tools()

for tool in kosis_tools:
    print("=====", tool.name)
    print(tool.description)                        # the text that the model reads about this tool
    for name in tool.args:
        print("   ", name, ":", tool.args[name].get("description", ""))   # each argument with its description

**Where to look.** The names of the tools, and the sentences of the descriptions that name another tool or say what the server does by itself.

### 5.2 Calling the tools without a model

A tool can be called directly, without a model, with arguments of our choice. Three calls try inputs that are not exact: a place name instead of its code, two regions written with a comma although the API takes `+`, and a code that the table does not have. The table is `DT_1B04006` (주민등록인구 by region and age), which the search tool of the server returns for 주민등록인구.

*Do:* run the cell. For each result, compare it with what the descriptions said, and find which part of it is data and which part is not.

In [ ]:
# ▶ RUN · example: read each line, then run
tools = {}
for tool in kosis_tools:
    tools[tool.name] = tool                        # find a tool by its name

calls = [
    ("kosis_region_code", {"orgId": "101", "tblId": "DT_1B04006", "region": "부산"}),
    ("kosis_get_data", {"orgId": "101", "tblId": "DT_1B04006", "objL1": "11,26", "objL2": "000", "itmId": "T2", "newEstPrdCnt": 1}),
    ("kosis_get_data", {"orgId": "101", "tblId": "DT_1B04006", "objL1": "11", "objL2": "0", "itmId": "T2", "newEstPrdCnt": 1}),
]
# objL1: region (11 = 서울, 26 = 부산), objL2: age (000 = all ages; "0" is not a code), itmId T2: 총인구수, newEstPrdCnt 1: the latest period

for name, args in calls:
    blocks = await tools[name].ainvoke(args)       # the call goes to the server without a model
    print("=====", name, args)
    print(blocks[0]["text"])                       # the result as the server sent it

**Where to look.** The lines before and after the numbers in each result, and the length of the result against the two numbers that we asked for.

### 5.3 The trace of an agent

The trace of a run is the list of all its messages. The cell prints the question, each tool call with its arguments, each tool result in full with its length, and the answer.

*Do:* run the cell. Compare the calls with the order that you wrote in 5.1, and for each part of the answer, find the tool result that asked for it.

In [ ]:
# ▶ RUN · example: read each line, then run
agent_kosis = create_agent(model, kosis_tools)
result = await agent_kosis.ainvoke({"messages": [HumanMessage("2025년 서울과 부산의 주민등록인구를 알려줘.")]})

for message in result["messages"]:
    if message.type == "human":
        print("===== QUESTION:", message.text)
    if message.type == "ai":
        for call in message.tool_calls:            # one AI message can ask for several tools
            print("===== TOOL CALL:", call["name"], call["args"])
        if message.text:
            print("===== ANSWER:")
            print(message.text)
    if message.type == "tool":
        print("===== TOOL RESULT:", message.name, "-", len(message.text), "characters")
        print(message.text)                        # the result in full, as the server sent it

**Where to look.** The lines of each result marked 🚨 or 💡, and the first and last lines of the answer.

### 5.4 The design of the server

*Do:* fill in the table from your notes. In the last column, write what our second server of Part 4 does.

| Design question | What the KOSIS server does | Evidence (the call and the part of its result) | Our server of Part 4 |
|---|---|---|---|
| How the work is split among the tools | | | |
| What the model is told about the use of the tools | | | |
| What a result carries besides the data | | | |
| What a call that is not exact gets back | | | |

1. Which of these choices follow the two principles of Part 3, and which do not?
2. Some instructions in the results are marked not to be shown to the user. What does this let a server do to the answer of an agent?